# Pneumonia Detection from Chest X-rays
Steps: download data → explore → load → augment → build model (MobileNetV2 transfer learning) → train → evaluate → predict.

**Colab: Runtime → Change runtime type → GPU (T4)** before running.

## 1. Download and unzip data

In [ ]:
!wget -q -O dataset.zip "https://d3ilbtxij3aepc.cloudfront.net/projects/CNN-PROJECT-7-11/Chest-Xray-2.zip"
!unzip -q -o dataset.zip -d dataset

## 2. Imports and settings

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

BASE = "/content/dataset/chest_xray"
IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42
tf.random.set_seed(SEED)

## 3. Explore: class counts and sample images

In [ ]:
for split in ["train", "test", "val"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        n = len(os.listdir(f"{BASE}/{split}/{cls}"))
        print(f"{split}/{cls}: {n}")

plt.figure(figsize=(12, 4))
for i, cls in enumerate(["NORMAL", "PNEUMONIA"]):
    folder = f"{BASE}/train/{cls}"
    for j, name in enumerate(os.listdir(folder)[:3]):
        plt.subplot(2, 3, i * 3 + j + 1)
        plt.imshow(plt.imread(os.path.join(folder, name)), cmap="gray")
        plt.title(cls); plt.axis("off")
plt.show()

## 4. Load data
The original `val` folder has only 16 images, which is too small to trust. So we take **20% of `train` as validation** instead. `test` is kept untouched for the final score.

In [ ]:
common = dict(image_size=IMG_SIZE, batch_size=BATCH, label_mode="binary", color_mode="rgb")

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/train", validation_split=0.2, subset="training", seed=SEED, **common)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/train", validation_split=0.2, subset="validation", seed=SEED, **common)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/test", shuffle=False, **common)

print(train_ds.class_names)   # NORMAL = 0, PNEUMONIA = 1

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 5. Handle class imbalance
There are ~3x more pneumonia images than normal ones. Class weights make the model pay more attention to the NORMAL class so it doesn't just predict "pneumonia" every time.

In [ ]:
n_normal = len(os.listdir(f"{BASE}/train/NORMAL"))
n_pneu = len(os.listdir(f"{BASE}/train/PNEUMONIA"))
total = n_normal + n_pneu
class_weight = {0: total / (2 * n_normal), 1: total / (2 * n_pneu)}
print(class_weight)

## 6. Build the model (data augmentation + pretrained MobileNetV2)

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False   # freeze pretrained layers first

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)   # scales pixels to [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.Precision(name="precision")])
model.summary()

## 7. Train (stage 1: train only the new top layers)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=10,
                    class_weight=class_weight, callbacks=callbacks)

## 8. Fine-tune (stage 2: unfreeze the last layers of MobileNetV2 with a small learning rate)

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:      # keep early layers frozen
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.Precision(name="precision")])

history_ft = model.fit(train_ds, validation_data=val_ds, epochs=8,
                       class_weight=class_weight, callbacks=callbacks)

## 9. Training curves

In [ ]:
acc = history.history["accuracy"] + history_ft.history["accuracy"]
val_acc = history.history["val_accuracy"] + history_ft.history["val_accuracy"]
loss = history.history["loss"] + history_ft.history["loss"]
val_loss = history.history["val_loss"] + history_ft.history["val_loss"]

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1); plt.plot(acc, label="train"); plt.plot(val_acc, label="val"); plt.title("Accuracy"); plt.legend()
plt.subplot(1, 2, 2); plt.plot(loss, label="train"); plt.plot(val_loss, label="val"); plt.title("Loss"); plt.legend()
plt.show()

## 10. Evaluate on the test set

In [ ]:
probs = model.predict(test_ds).ravel()
y_pred = (probs > 0.5).astype(int)
y_true = np.concatenate([y.numpy() for _, y in test_ds]).ravel().astype(int)

print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"]))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["NORMAL", "PNEUMONIA"]).plot(cmap="Blues")
plt.show()

Note: in medical screening, missing a pneumonia case (false negative) is worse than a false alarm. If **recall** for PNEUMONIA is low, lower the threshold (e.g. `probs > 0.4`).

## 11. Save the model

In [ ]:
model.save("pneumonia_model.keras")

## 12. Predict on a single X-ray
Change `img_path` to any image (e.g. upload your own).

In [ ]:
def predict_xray(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    arr = tf.expand_dims(tf.keras.utils.img_to_array(img), 0)
    p = float(model.predict(arr, verbose=0)[0][0])
    label = "PNEUMONIA" if p > 0.5 else "NORMAL"
    conf = p if p > 0.5 else 1 - p
    plt.imshow(img); plt.axis("off"); plt.title(f"{label} ({conf:.1%} confident)"); plt.show()
    return label

folder = f"{BASE}/test/PNEUMONIA"
predict_xray(os.path.join(folder, os.listdir(folder)[0]))
folder = f"{BASE}/test/NORMAL"
predict_xray(os.path.join(folder, os.listdir(folder)[0]))